In [8]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from surprise import Reader, Dataset, SVD
from surprise.model_selection import train_test_split

In [9]:
movie_data = pd.read_csv("cleaned_movie_data.csv")
movie_data.head()

,userId,movieId,rating,title,genres
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


## Content-Based part

In [10]:
# Keep unique movies only
movie_content = movie_data[['movieId', 'title', 'genres']].drop_duplicates()

In [11]:
# Convert genres into numerical vectors
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movie_content['genres'])

In [12]:
# Compute similarity between movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [13]:
# Create mapping between movie titles and indices
indices = pd.Series(movie_content.index,index=movie_content['title']).drop_duplicates()

## Collaborative part

In [14]:
# Define rating scale
reader = Reader(rating_scale=(0.5, 5))

# Convert dataframe into surprise dataset
data = Dataset.load_from_df(movie_data[['userId', 'movieId', 'rating']],reader)

In [15]:
# Split dataset into train and test sets
trainset, testset = train_test_split(data,test_size=0.2,random_state=42)

In [16]:
# Create and train SVD model
model = SVD()

model.fit(trainset)

## Hybrid_Recommendation function

In [17]:
def hybrid_recommendation(user_id, title, top_n=10):

    # Get movie index
    idx = indices[title]

    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort movies based on similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top similar movies
    sim_scores = sim_scores[1:top_n+1]

    recommendations = []

    for i, score in sim_scores:

        # Get movie id
        movie_id = movie_content.iloc[i]['movieId']

        # Get predicted rating from SVD
        predicted_rating = model.predict(user_id, movie_id).est

        # Hybrid score
        hybrid_score = (0.7 * predicted_rating) + (0.3 * score)

        recommendations.append((
            movie_content.iloc[i]['title'],
            hybrid_score
        ))

    # Sort final recommendations
    recommendations = sorted(
        recommendations,
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations

In [18]:
hybrid_recommendation(1, 'Toy Story (1995)')

[('Toy Story 2 (1999)', 3.599519390494175),
 ('Monsters, Inc. (2001)', 3.515825921299731),
 ('Moana (2016)', 3.39022673691661),
 ("Emperor's New Groove, The (2000)", 3.3816534670636442),
 ('Asterix and the Vikings (Astérix et les Vikings) (2006)',
  3.2523348458150925),
 ('Tale of Despereaux, The (2008)', 3.231172514744277),
 ('The Good Dinosaur (2015)', 3.228587501911096),
 ('Antz (1998)', 3.1465155214256337),
 ('Adventures of Rocky and Bullwinkle, The (2000)', 3.0106431011090202),
 ('Shrek the Third (2007)', 2.8814956765645934)]

In [20]:
hybrid_recommendation(5, 'Heat (1995)')

[('Kill Bill: Vol. 1 (2003)', 3.0222642869962533),
 ('Bourne Ultimatum, The (2007)', 2.9159696029085618),
 ('Bourne Supremacy, The (2004)', 2.846759159280319),
 ('Ronin (1998)', 2.7582686778755345),
 ('Batman (1989)', 2.739968898672571),
 ('Die Hard: With a Vengeance (1995)', 2.5947057688992663),
 ('Natural Born Killers (1994)', 2.5861265245937437),
 ('xXx (2002)', 2.3489079078662405),
 ('Net, The (1995)', 2.318945592772404),
 ('Shaft (2000)', 2.2661648865618655)]

In [19]:
## Hybrid Recommendation System

#This function combines content-based similarity scores with collaborative filtering predicted ratings using a weighted average.